# Lab Tracking Day22 — chạy trên Colab (GPU)
Notebook tự động: cài đặt → kiểm tra dữ liệu → baseline → so sánh tracker → quét conf/iou → chạy nộp bài → chấm video_1.

In [ ]:
# 1. Cài đặt + tải dữ liệu
%cd /content
!rm -rf /content/repo
!git clone -q https://github.com/nguyenducdong22/K4-Track4-Day22-NguyenDucDong-2A202602367-Tracking-Student.git /content/repo
# boxmot 10.0.42 ghim numpy==1.23.1 (không build được trên Python 3.12) -> cài --no-deps rồi cài phụ thuộc riêng
!pip install -q "ultralytics>=8.4" "lapx>=0.5.4" filterpy ftfy GitPython loguru yacs gdown
!pip install -q --no-deps "boxmot==10.0.42"
![ -d /content/TrackEval ] || git clone -q https://github.com/JonathonLuiten/TrackEval.git /content/TrackEval
!pip install -q --no-deps -e /content/TrackEval
import os, glob, io
FILE_ID = '1UeVPQd6j5pSzxoJDcKJrerT9SL3vJLDt'
ZIP = '/content/data.zip'
if not os.path.exists(ZIP) or os.path.getsize(ZIP) < 1e6:
    os.system(f'gdown -q {FILE_ID} -O {ZIP}')
if not os.path.exists(ZIP) or os.path.getsize(ZIP) < 1e6:
    # gdown bị giới hạn lượt tải -> tải bằng tài khoản Google của bạn (Drive API)
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    svc = build('drive', 'v3')
    req = svc.files().get_media(fileId=FILE_ID, supportsAllDrives=True)
    with io.FileIO(ZIP, 'wb') as fh:
        dl = MediaIoBaseDownload(fh, req, chunksize=50*1024*1024)
        done = False
        while not done:
            st, done = dl.next_chunk()
            print(f'Đang tải {int(st.progress()*100)}%')
print('data.zip:', round(os.path.getsize(ZIP)/1e6, 1), 'MB')
![ -d /content/lab_raw ] || unzip -q -o /content/data.zip -d /content/lab_raw
cand = [os.path.dirname(p) for p in glob.glob('/content/lab_raw/**/video_1', recursive=True)]
LAB = cand[0]
os.environ['LAB_DATA'] = LAB
%cd /content/repo
print('LAB_DATA =', LAB)
!python -c "import boxmot, numpy; print('boxmot', boxmot.__version__, 'numpy', numpy.__version__)"
!nvidia-smi --query-gpu=name --format=csv,noheader


In [ ]:
# 2. CP1: kiểm tra dữ liệu + YOLO một frame (chưa có ID)
LAB = [os.path.dirname(p) for p in glob.glob('/content/lab_raw/**/video_1', recursive=True) if '__MACOSX' not in p][0]
os.environ['LAB_DATA'] = LAB
print('LAB_DATA =', LAB)
!python scripts/check_data.py --lab-data-root "$LAB_DATA"
from ultralytics import YOLO
from pathlib import Path
import cv2
f0 = sorted(Path(LAB, 'video_1', 'img1').glob('*.jpg'))[0]
det = YOLO('yolo26n.pt')
for c in (0.15, 0.3, 0.5):
    r = det.predict(cv2.imread(str(f0)), conf=c, iou=0.5, imgsz=640, classes=[0], verbose=False)[0]
    print(f'conf={c}: {len(r.boxes)} hộp người (chưa có ID)')
for v in ['video_1','video_2','video_3','video_4','video_5']:
    print(v, len(list(Path(LAB, v, 'img1').glob('*.jpg'))), 'frame')


In [ ]:
# 3. Hàm tiện ích: chạy tracker, thống kê track, chấm video_1
import subprocess, sys, re, json, time, shutil
import numpy as np, pandas as pd
TRACKERS = ['bytetrack','ocsort','botsort','strongsort','deepocsort']
LOG = []
def stats(p):
    try:
        a = np.loadtxt(p, delimiter=',', ndmin=2)
    except Exception:
        a = np.empty((0,10))
    if a.size == 0:
        return dict(boxes=0, ids=0, box_per_frame=0, mean_len=0, short_frac=0, ids_per_100box=0)
    ids, cnt = np.unique(a[:,1], return_counts=True)
    nf = a[:,0].max()
    return dict(boxes=len(a), ids=len(ids), box_per_frame=round(len(a)/nf,2),
                mean_len=round(cnt.mean(),1), short_frac=round((cnt<10).mean(),3),
                ids_per_100box=round(100*len(ids)/len(a),3))
def run(v, trk, conf=0.3, iou=0.5, out='runs/thu', maxf=0, video=False):
    cmd = [sys.executable,'scripts/run_tracking.py','--source',f'{LAB}/{v}/img1','--seq-name',v,
           '--tracker',trk,'--conf',str(conf),'--iou',str(iou),'--out',out,'--device','cuda:0']
    if maxf: cmd += ['--max-frames', str(maxf)]
    if video: cmd += ['--save-video']
    t = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-3000:]); raise RuntimeError('run_tracking lỗi')
    s = stats(f'{out}/{v}.txt')
    s.update(video=v, tracker=trk, conf=conf, iou=iou, frames=maxf or 'all', sec=round(time.time()-t,1))
    return s
def eval_v1(sub, name):
    r = subprocess.run([sys.executable,'scripts/evaluate_practice.py','--trackeval-root','/content/TrackEval',
                        '--lab-data-root',LAB,'--submission',sub,'--run-name',name], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:]); raise RuntimeError('eval lỗi')
    summ = glob.glob(f'/content/TrackEval/data/trackers/mot_challenge/*/{name}/pedestrian_summary.txt')[0]
    h, v = open(summ).read().split('\n')[:2]
    d = dict(zip(h.split(), map(float, v.split())))
    return dict(HOTA=d['HOTA'], DetA=d['DetA'], AssA=d['AssA'], MOTA=d['MOTA'], IDF1=d['IDF1'],
                IDSW=int(d['IDSW']), FP=int(d['CLR_FP']), FN=int(d['CLR_FN'])), r.stdout
print('ok')


In [ ]:
# 4. CP2: baseline ByteTrack 150 frame trên video_1
b = run('video_1','bytetrack',0.3,0.5,out='runs/thu_nhanh',maxf=150,video=True); print(b)


In [ ]:
# 5. CP3 — video_1 (có nhãn): so sánh 5 tracker (đủ frame), rồi quét conf / iou
os.makedirs('ket_qua', exist_ok=True)
# Phòng hờ: TrackEval cũ dùng np.float/np.int (NumPy 2 đã bỏ) và gói dữ liệu thiếu eval_config.json
!find /content/TrackEval/trackeval -name '*.py' | xargs sed -i -E 's/np\.float\b/float/g; s/np\.int\b/int/g; s/np\.bool\b/bool/g'
cfg = Path(LAB, 'video_1', 'eval_config.json')
if not cfg.exists(): cfg.write_text(json.dumps({'benchmark': 'LAB', 'split': 'train'}))
rows = []
for trk in TRACKERS:
    s = run('video_1', trk, 0.3, 0.5, out=f'runs/v1_{trk}')
    m, _ = eval_v1(f'runs/v1_{trk}/video_1.txt', f'v1_{trk}'); s.update(m); rows.append(s); print(s)
best = max(rows, key=lambda r: r['HOTA'])['tracker']
for conf, iou in [(0.15,0.5),(0.5,0.5),(0.3,0.4),(0.3,0.7)]:
    tag = f'v1_{best}_c{conf}_i{iou}'
    s = run('video_1', best, conf, iou, out=f'runs/{tag}')
    m, _ = eval_v1(f'runs/{tag}/video_1.txt', tag); s.update(m); rows.append(s); print(s)
df1 = pd.DataFrame(rows); df1.to_csv('ket_qua/sweep_video_1.csv', index=False)
display(df1[['tracker','conf','iou','HOTA','DetA','AssA','MOTA','IDF1','IDSW','FP','FN','ids','sec']])


In [ ]:
# 6. CP3 — video_2..video_5 (không nhãn): so sánh 5 tracker trên 300 frame đầu bằng thống kê track
# Chỉ số proxy: ids_per_100box thấp = ít tạo ID mới / ít đứt track; short_frac = tỉ lệ track < 10 frame.
MAXF = 300
CHOICE = {}
allrows = []
for v in ['video_2','video_3','video_4','video_5']:
    rows = [run(v, trk, 0.3, 0.5, out=f'runs/{v}_{trk}', maxf=MAXF) for trk in TRACKERS]
    cov = max(r['box_per_frame'] for r in rows)
    ok = [r for r in rows if r['box_per_frame'] >= 0.9*cov]
    best = min(ok, key=lambda r: r['ids_per_100box'])['tracker']
    base = [r for r in rows if r['tracker']==best][0]
    for conf, iou in [(0.15,0.5),(0.5,0.5),(0.3,0.4),(0.3,0.7)]:
        rows.append(run(v, best, conf, iou, out=f'runs/{v}_{best}_c{conf}_i{iou}', maxf=MAXF))
    # chọn conf trước (giữ iou=0.5), rồi iou (giữ conf đã chọn = 0.3 nếu conf đổi thì giữ iou 0.5)
    sweep = [r for r in rows if r['tracker']==best]
    good = [r for r in sweep if 0.9*base['box_per_frame'] <= r['box_per_frame'] <= 1.25*base['box_per_frame']]
    pick = min(good, key=lambda r: r['ids_per_100box'])
    CHOICE[v] = (best, pick['conf'], pick['iou'])
    df = pd.DataFrame(rows); df.to_csv(f'ket_qua/sweep_{v}.csv', index=False); allrows += rows
    print(v, '->', CHOICE[v])
    display(df[['tracker','conf','iou','boxes','ids','box_per_frame','mean_len','short_frac','ids_per_100box','sec']])


In [ ]:
# 7. Chọn cấu hình video_1 theo HOTA và chạy NỘP BÀI đủ frame cho cả 5 video
b1 = df1.loc[df1['HOTA'].idxmax()]
CHOICE['video_1'] = (b1['tracker'], float(b1['conf']), float(b1['iou']))
shutil.rmtree('runs/nop_bai', ignore_errors=True)
final = []
for v in ['video_1','video_2','video_3','video_4','video_5']:
    trk, c, i = CHOICE[v]
    s = run(v, trk, c, i, out='runs/nop_bai', video=True); final.append(s); print(s)
m1, out1 = eval_v1('runs/nop_bai/video_1.txt', 'nop_bai_video1')
open('ket_qua/eval_video_1.txt','w').write(out1)
pd.DataFrame(final).to_csv('ket_qua/final_configs.csv', index=False)
print(out1[-3500:])
print('METRICS video_1:', m1)
print('CHOICE_JSON=' + json.dumps(CHOICE))


In [ ]:
# 8. Ảnh kiểm tra bằng mắt: 4 frame có vẽ ID từ mỗi video nộp
import matplotlib.pyplot as plt
for v in ['video_1','video_2','video_3','video_4','video_5']:
    cap = cv2.VideoCapture(f'runs/nop_bai/{v}_preview.mp4'); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fig, ax = plt.subplots(1, 4, figsize=(20, 4))
    for k, idx in enumerate(np.linspace(0, n-1, 4).astype(int)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx); ok, fr = cap.read()
        if ok: ax[k].imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        ax[k].set_title(f'{v} frame {idx+1}'); ax[k].axis('off')
    plt.tight_layout(); plt.savefig(f'ket_qua/{v}_frames.jpg', dpi=60); plt.show()


In [ ]:
# 9. Đóng gói kết quả và tải về
!cd /content/repo && zip -q -r /content/ket_qua_lab22.zip runs/nop_bai/*.txt ket_qua
from google.colab import files
files.download('/content/ket_qua_lab22.zip')
